# ⚠️ 하루마다 업데이트?

## 🔍 기업마당 금융 탭 크롤링 

In [ ]:
import os
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import re
from urllib.parse import urlparse, parse_qs
import requests
import zipfile


import os
from dotenv import load_dotenv
import pickle
import faiss
import numpy as np
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import re
import os
from langchain.document_loaders import PDFPlumberLoader, TextLoader
from langchain_community.vectorstores import FAISS

from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableMap
from langchain.callbacks import get_openai_callback



# 셀레니움 옵션 설정
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
driver = webdriver.Chrome(options=options)

In [29]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")  # 환경변수 설정

In [ ]:
# 대상 URL
base_url = (
    "https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/list.do"
    "?hashCode=01&rowsSel=6&cat=&article_seq=&pblancId=&schJrsdCodeTy="
    "&schWntyAt=&schAreaDetailCodes=&schEndAt=N&orderGb=&sort="
    "&condition=searchPblancNm&condition1=AND&preKeywords=&keyword=&rows=15"
)
detail_base = "https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/"

## 🆕 새로 업데이트된 공고 추가하기

In [ ]:
import pandas as pd
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# 셀레니움 설정
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
driver = webdriver.Chrome(options=options)

# 기본 URL 설정
base_url = "https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/list.do"
detail_base = "https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/"

# 기존 데이터 로드 (없으면 빈 DataFrame 생성)
csv_path = "bizinfo_지원사업_공고목록.csv"
try:
    existing_df = pd.read_csv(csv_path)
    existing_set = set(zip(existing_df["제목"], existing_df["링크"]))
    print(f"📂 기존 데이터: {len(existing_df)}건")
except FileNotFoundError:
    existing_df = pd.DataFrame(columns=["제목", "링크"])
    existing_set = set()
    print("📂 기존 CSV 파일 없음. 새로 생성합니다.")

# 새로운 데이터 수집 리스트
new_data = []

# HTML 파싱 함수
def parse_current_page_html(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = soup.select("div.table_Type_1 > table > tbody > tr")
    
    for row in rows:
        title_tag = row.select_one("td.txt_l a")
        if title_tag:
            title = title_tag.get_text(strip=True)
            href = title_tag.get("href")
            full_link = detail_base + href if href else None

            if (title, full_link) not in existing_set:
                new_data.append({"제목": title, "링크": full_link})
                print(f"🆕 신규 공고 발견: {title}")

# 페이지 수 설정
max_pages = 2  # 겹치는 거 많을 수도 있으니까 일단 2페이지만 새로 가져와보기
print(f"▶ 총 수집 페이지 수: {max_pages}")

for page in range(1, max_pages + 1):
    driver.get(f"{base_url}?cpage={page}")
    time.sleep(2)
    parse_current_page_html(driver.page_source)
    print(f"✅ {page}페이지 완료")

driver.quit()

# 새 공고를 기존 데이터에 추가 후 저장
if new_data:
    # 📊 신규 공고만 따로 DataFrame으로 빼놓기 (저장은 하지 않음)
    new_data_df = pd.DataFrame(new_data)
    print(f"📊 신규 공고 DataFrame 생성: {len(new_data_df)}건")
    
    new_df = pd.DataFrame(new_data)
    updated_df = pd.concat([existing_df, new_df], ignore_index=True)
    updated_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"✅ 신규 {len(new_df)}건 추가 저장 완료")
else:
    print("📭 추가된 새로운 공고 없음.")

📂 기존 데이터: 45건
▶ 총 수집 페이지 수: 2
✅ 1페이지 완료
✅ 2페이지 완료
📭 추가된 새로운 공고 없음.


## 📁 공고에서 첨부파일 다운로드 받기

In [10]:
# 🔽 CSV 파일 경로 지정 (수정 필요)
csv_path = "bizinfo_지원사업_공고목록.csv"

# CSV 불러오기
df = pd.read_csv(csv_path)

def extract_pblanc_id(link):
    if isinstance(link, str) and "pblancId=" in link:
        parsed = urlparse(link)
        query = parse_qs(parsed.query)
        return query.get("pblancId", [None])[0]
    return None

df["pblanc_id"] = df["링크"].apply(extract_pblanc_id)
df = df.dropna(subset=["pblanc_id"])
df.to_csv("bizinfo_지원사업_공고목록.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ✅ ZIP 깨진 한글 파일명 복구 & 압축 해제 함수
def extract_zip_with_encoding(zip_path, extract_to, encoding="euc-kr"):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for zip_info in zip_ref.infolist():
            try:
                decoded_name = zip_info.filename.encode('cp437').decode(encoding)
                zip_info.filename = decoded_name  # 복구된 이름으로 설정
                zip_ref.extract(zip_info, path=extract_to)
                print(f"📄 추출됨: {decoded_name}")
            except Exception as e:
                print(f"❌ 파일 추출 실패: {zip_info.filename} ({e})")

In [ ]:
# ✅ 첨부파일 다운로드 함수
def download_files(pblanc_id):
    save_dir = os.path.join(os.getcwd(), "download(support)", pblanc_id)
    os.makedirs(save_dir, exist_ok=True)

    url = f"https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/view.do?pblancId={pblanc_id}"

    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        links = soup.select("a.icon_download")
        if not links:
            print(f"❌ 첨부파일 없음: {pblanc_id}")
            return

        for link in links:
            file_name = link["title"].replace("첨부파일 ", "").replace(" 다운로드", "").strip()
            file_url = "https://www.bizinfo.go.kr" + link["href"]

            try:
                file_res = requests.get(file_url)
                file_res.raise_for_status()

                file_path = os.path.join(save_dir, file_name)
                with open(file_path, "wb") as f:
                    f.write(file_res.content)

                # ✅ ZIP 처리
                if file_name.lower().endswith(".zip"):
                    try:
                        extract_zip_with_encoding(file_path, save_dir, encoding="euc-kr")
                        os.remove(file_path)
                        print(f"✅ ZIP 압축 해제 및 삭제: {file_path}")
                    except zipfile.BadZipFile:
                        print(f"⚠️ 손상된 ZIP 파일: {file_path}")
                else:
                    print(f"✅ 저장 완료: {file_path}")

            except Exception as e:
                print(f"❌ 다운로드 실패: {file_name} ({e})")

    except Exception as e:
        print(f"❌ 요청 실패: {pblanc_id} ({e})")

In [ ]:
# 🔍 pblanc_id 추출 함수
def extract_pblanc_id(link):
    if isinstance(link, str) and "pblancId=" in link:
        parsed = urlparse(link)
        query = parse_qs(parsed.query)
        return query.get("pblancId", [None])[0]
    return None

# ✅ new_data_df에서 pblanc_id 추출
new_data_df["pblanc_id"] = new_data_df["링크"].apply(extract_pblanc_id)
new_data_df = new_data_df.dropna(subset=["pblanc_id"])

# ✅ 각 pblanc_id에 대해 첨부파일 다운로드 실행
for pblanc_id in new_data_df["pblanc_id"].unique():
    print(f"\n⬇️ 첨부파일 다운로드 시작: {pblanc_id}")
    download_files(pblanc_id)

## ✏️ 새로운 공고에 한해서 다운된 HWP 파일들 모두 변환해서 JSON 파일로 다시 저장

In [ ]:
import os
import olefile
import shutil
import json
import uuid

In [ ]:
# ✅ 이미 생성된 new_data_df 사용 (pblanc_id 포함)
new_ids = new_data_df["pblanc_id"].dropna().unique().tolist()

# ✅ .hwp → 텍스트 추출 함수
def extract_text_from_hwp(file_path):
    try:
        temp_path = os.path.join(os.getcwd(), "temp.hwp")
        shutil.copy(file_path, temp_path)

        with olefile.OleFileIO(temp_path) as f:
            encoded_text = f.openstream('PrvText').read()
            decoded_text = encoded_text.decode('utf-16')

        os.remove(temp_path)
        return decoded_text

    except Exception as e:
        print(f"❌ 변환 실패: {file_path} - {e}")
        return None

# ✅ 신규 공고(pblanc_id) 폴더에서만 변환 수행
def convert_new_hwp_to_json(base_dir, new_ids):
    for pblanc_id in new_ids:
        folder_path = os.path.join(base_dir, pblanc_id)
        if not os.path.isdir(folder_path):
            print(f"🚫 폴더 없음: {folder_path}")
            continue

        for file in os.listdir(folder_path):
            if file.lower().endswith(".hwp"):
                full_path = os.path.join(folder_path, file)
                text = extract_text_from_hwp(full_path)
                if text:
                    json_name = os.path.splitext(file)[0] + ".json"
                    json_path = os.path.join(folder_path, json_name)

                    with open(json_path, "w", encoding="utf-8") as f:
                        json.dump({"file_name": file, "text": text}, f, ensure_ascii=False, indent=2)

                    print(f"✅ 변환 완료: {pblanc_id}/{file} → {json_name}")

# ✅ 실행
base_dir = os.path.join(os.getcwd(), "download(support)")
convert_new_hwp_to_json(base_dir, new_ids)

현재 폴더: c:\Users\HY\Desktop\KB AI Challenge\download(support)
현재 폴더: c:\Users\HY\Desktop\KB AI Challenge\download(support)\PBLN_000000000112804
현재 폴더: c:\Users\HY\Desktop\KB AI Challenge\download(support)\PBLN_000000000112806
✅ 변환 완료: 2025+하반기+소상공인시장진흥자금+신청서+등(부동산+담보용).hwp → 2025+하반기+소상공인시장진흥자금+신청서+등(부동산+담보용).json
✅ 변환 완료: 2025+하반기+소상공인진흥자금+공고문.hwp → 2025+하반기+소상공인진흥자금+공고문.json


## 📦 벡터스토어 저장

In [ ]:
# 제외 키워드 : '지원서', '사업계획서', '계획서', '서식', '양식', '신청서', '동의서', '서약서', '신청서', '의향서', '확인서'
# 저장된 공고는 csv 파일의 '벡터스토어 저장'부분에 TRUE 해놓기
# 벡터스토어에 저장할 때 메타데이터에 공고명이랑 pblanc_id도 저장해놓자...

In [ ]:
# 제외할 키워드 목록
EXCLUDE_KEYWORDS = ['지원서', '사업계획서', '계획서', '서식', '양식', '신청서', 
                    '동의서', '서약서', '의향서', '확인서']

# 마침표 뒤가 아닌 줄바꿈 제거 함수
def remove_newlines_except_after_period(text):
    return re.sub(r'(?<!\.)(\n|\r\n)', ' ', text)

# 파일 로딩 함수 (PDF, JSON)
def load_document(file_path, folder_name, title):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        loader = PDFPlumberLoader(file_path)
    elif ext == ".json":
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        return []

    docs = loader.load()
    for doc in docs:
        doc.page_content = remove_newlines_except_after_period(doc.page_content)
        doc.metadata["folder_name"] = folder_name
        doc.metadata["file_type"] = ext
        doc.metadata["title"] = title  # ✅ 공고명
    return docs


# 전체 벡터스토어 생성 및 저장 함수
def save_vectorstore_for_new_items(base_dir, vectorstore_base_dir, csv_path, openai_api_key, new_data_df):
    df = pd.read_csv(csv_path)
    if "벡터스토어 저장" not in df.columns:
        df["벡터스토어 저장"] = False

    # ✅ new_data_df에서 pblanc_id만 추출
    new_ids = new_data_df["pblanc_id"].dropna().astype(str).unique()

    for folder_name in new_ids:
        folder_path = os.path.join(base_dir, folder_name)
        if not os.path.isdir(folder_path):
            print(f"🚫 폴더 없음: {folder_name}")
            continue

        # 🔍 공고명(title) 가져오기
        match_row = df[df["pblanc_id"].astype(str) == folder_name]
        if match_row.empty:
            print(f"❌ 공고명 찾을 수 없음: {folder_name}")
            continue
        title = match_row.iloc[0]["제목"]

        all_docs = []
        for file_name in os.listdir(folder_path):
            if not (file_name.endswith(".pdf") or file_name.endswith(".json")):
                continue
            if any(keyword in file_name for keyword in EXCLUDE_KEYWORDS):
                continue

            file_path = os.path.join(folder_path, file_name)
            docs = load_document(file_path, folder_name, title)
            all_docs.extend(docs)

        if all_docs:
            print(f"📄 {folder_name}: {len(all_docs)}개 문서 처리 중...")
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
            split_docs = text_splitter.split_documents(all_docs)
            embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)
            vectorstore = FAISS.from_documents(split_docs, embeddings)

            folder_vector_path = os.path.join(vectorstore_base_dir, folder_name)
            os.makedirs(folder_vector_path, exist_ok=True)
            vectorstore.save_local(folder_vector_path)

            with open(os.path.join(folder_vector_path, "documents.pkl"), "wb") as f:
                pickle.dump(split_docs, f)

            df.loc[df["pblanc_id"].astype(str) == folder_name, "벡터스토어 저장"] = True

    df.to_csv(csv_path, index=False)
    print("✅ 신규 벡터스토어 저장 및 CSV 업데이트 완료.")

In [2]:
import pandas as pd
pd.read_csv("bizinfo_지원사업_공고목록.csv")

,제목,링크,pblanc_id,벡터스토어 저장
0,[인천] 2025년 하반기 소상공인시장진흥자금 융자 계획 공고,https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C...,PBLN_000000000112806,True
1,[대구] 달성군 2025년 중소기업 경영안정자금(이차보전) 지원계획 변경 공고,https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C...,PBLN_000000000112804,False
2,[제주] 서귀포시 2025년 4차 사회적기업 사회보험료 지원사업 모집 공고,https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C...,PBLN_000000000112785,False
3,[대구] 남구 2025년 소상공인 경영안정자금 지원 사업 공고,https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C...,PBLN_000000000112802,False
4,[인천] 동구 영세 소상공인 경영회복지원금 신청기간 연장 공고,https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C...,PBLN_000000000112773,False
...,...,...,...,...
246,[충북] 2025년 중소기업육성자금 융자(이차보전) 지원계획 공고,https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C...,PBLN_000000000103387,False
247,[경북] 2025년 창업 및 경쟁력강화 사업자금 지원 계획 공고,https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C...,PBLN_000000000103381,False
248,2025년 중소벤처기업부 소관 소상공인 정책자금 융자계획 공고,https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C...,PBLN_000000000103358,False
249,2025년 중소기업 정책자금 융자계획 공고,https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C...,PBLN_000000000103364,False


In [ ]:
base_dir = os.path.join(os.getcwd(), "download(support)")
vectorstore_base_dir = os.path.join(os.getcwd(), "embeddings")
csv_path = os.path.join(os.getcwd(), "bizinfo_지원사업_공고목록.csv")

save_vectorstore_for_new_items(
    base_dir=base_dir,
    vectorstore_base_dir=vectorstore_base_dir,
    csv_path=csv_path,
    #openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_key=openai_api_key,
    new_data_df=new_data_df  # 🆕 신규 공고 DataFrame
)

In [30]:
base_dir = os.path.join(os.getcwd(), "download(support)")
vectorstore_base_dir = os.path.join(os.getcwd(), "embeddings")
csv_path = os.path.join(os.getcwd(), "bizinfo_지원사업_공고목록.csv")

save_vectorstore_from_folders(base_dir, vectorstore_base_dir, csv_path, openai_api_key)

📄 PBLN_000000000112806: 1개 문서 처리 중...
✅ 모든 벡터스토어 저장 및 CSV 업데이트 완료.
